In [ ]:
# Upload the dataset
from google.colab import files

uploaded = files.upload()


In [ ]:
# Import the required libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import OneHotEncoder
from google.colab import files


In [ ]:
# ============================================================
# Reproducibility and GPU check
# ============================================================
import random

SEED = 42


def set_seed(seed=SEED):
    """Fix every random source used in this notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device     :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU is active. In Colab: Runtime > Change runtime type > "
        "Hardware accelerator = GPU (T4/L4), then restart the session and "
        "run the notebook from the first cell."
    )


In [ ]:
# Load the CSV file into a pandas DataFrame
file_name = 'df_train.csv'
df = pd.read_csv(file_name)

df.info()


## Option A - discrete-value schema

In [ ]:
# ================================================================
# OPTION A - DISCRETE-VALUE SCHEMA
# ----------------------------------------------------------------
# A generator that ends in a Sigmoid emits a continuous value in
# [0, 1] for EVERY column, including the columns that can only hold
# a small number of valid levels. A synthetic record may therefore
# contain "Smoking Status = 0.63", which no real patient can have.
#
# This cell derives from df_train itself which columns are discrete
# and which levels each one is allowed to take. Nothing is
# hard-coded, so the rule stays correct if the preprocessing ever
# changes.
#
# Expected grouping for this dataset (36 features):
#     3  ordinal   : Physical Activity, Socioeconomic Factors,
#                    Alcohol Consumption            -> {0.0, 0.5, 1.0}
#    16  binary    : Dietary Habits ... Early Onset Symptoms
#                                                   -> {0.0, 1.0}
#     4  one-hot   : Urine Test_*                   -> exactly one 1
#     1  ordinal   : Neurological Assessments       -> {0.0, 0.5, 1.0}
#                    (numeric in the raw file, so the original code
#                     treated it as continuous - this is the column
#                     whose KDE plot was visibly wrong)
#    12  continuous: Age, BMI, Insulin Levels, ...
# ================================================================
import numpy as np
import pandas as pd

SCHEMA_TARGET_COL   = "Target"
MAX_DISCRETE_LEVELS = 12          # at most this many unique values => discrete
ONEHOT_PREFIXES     = ["Urine Test_"]


def build_schema(df_reference, target_col=SCHEMA_TARGET_COL):
    """Describe every feature column of the real training data."""
    schema = {"discrete": {}, "continuous": {}, "onehot_groups": {}}
    features = [c for c in df_reference.columns if c != target_col]

    for prefix in ONEHOT_PREFIXES:
        members = sorted(c for c in features if c.startswith(prefix))
        if members:
            schema["onehot_groups"][prefix] = members

    grouped = {c for m in schema["onehot_groups"].values() for c in m}

    for col in features:
        if col in grouped:
            continue
        levels = np.unique(df_reference[col].dropna().to_numpy(dtype=float))
        if len(levels) <= MAX_DISCRETE_LEVELS:
            schema["discrete"][col] = np.sort(levels)
        else:
            schema["continuous"][col] = (
                float(df_reference[col].min()),
                float(df_reference[col].max()),
            )
    return schema


def apply_schema(df_synth, schema, target_col=SCHEMA_TARGET_COL):
    """Return a copy of df_synth in which every value is legal."""
    out = df_synth.copy()

    # 1. snap each discrete column to its nearest legal level
    for col, levels in schema["discrete"].items():
        if col not in out.columns:
            continue
        values = out[col].to_numpy(dtype=float)
        idx = np.abs(values[:, None] - levels[None, :]).argmin(axis=1)
        out[col] = levels[idx]

    # 2. reduce each one-hot group to a single 1 (arg-max wins)
    for members in schema["onehot_groups"].values():
        members = [c for c in members if c in out.columns]
        if not members:
            continue
        block = out[members].to_numpy(dtype=float)
        hard = np.zeros_like(block)
        hard[np.arange(len(block)), block.argmax(axis=1)] = 1.0
        out[members] = hard

    # 3. keep continuous columns inside the range seen in the real data
    for col, (lo, hi) in schema["continuous"].items():
        if col in out.columns:
            out[col] = out[col].clip(lo, hi)

    return out


def validity_report(df, schema, target_col=SCHEMA_TARGET_COL, label=""):
    """Percentage of rows holding a legal value, per discrete column."""
    rows = []
    for col, levels in schema["discrete"].items():
        if col not in df.columns:
            continue
        ok = np.isin(df[col].to_numpy(dtype=float), levels).mean() * 100.0
        rows.append((col, "discrete", len(levels), round(ok, 2)))

    for prefix, members in schema["onehot_groups"].items():
        members = [c for c in members if c in df.columns]
        if not members:
            continue
        block = df[members].to_numpy(dtype=float)
        ok = (np.isin(block, [0.0, 1.0]).all(axis=1)
              & (block.sum(axis=1) == 1.0)).mean() * 100.0
        rows.append((prefix + "*", "one-hot", len(members), round(ok, 2)))

    rep = pd.DataFrame(rows, columns=["Column", "Type", "Levels", "Valid (%)"])
    rep = rep.sort_values("Valid (%)").reset_index(drop=True)
    print("\n--- Validity report " + str(label) + " ---")
    print("Columns fully valid : {} / {}".format((rep["Valid (%)"] == 100).sum(), len(rep)))
    print("Mean validity       : {:.2f}%".format(rep["Valid (%)"].mean()))
    display(rep)
    return rep


# Build the schema from the real training data
_df_reference = pd.read_csv("df_train.csv")
SCHEMA = build_schema(_df_reference)

print("Discrete columns   :", len(SCHEMA["discrete"]))
print("One-hot groups     :", len(SCHEMA["onehot_groups"]))
print("Continuous columns :", len(SCHEMA["continuous"]))
print("\nDiscrete columns and their legal levels:")
for _c, _l in SCHEMA["discrete"].items():
    print("  {:<32} {}".format(_c, np.round(_l, 4).tolist()))


In [ ]:
df_cwgan = df.copy()
TARGET_COL = "Target"

X = df_cwgan.drop(columns=[TARGET_COL])
y = df_cwgan[[TARGET_COL]]

n_features = X.shape[1]

# One-hot encoded target
ohe = OneHotEncoder(sparse_output=False)
y_ohe = ohe.fit_transform(y)

X_np = X.values.astype("float32")  # assumed to be already scaled to [0, 1]
y_np = y_ohe.astype("float32")

input_dim = n_features
cond_dim = y_np.shape[1]


In [ ]:
# Main configuration
EPOCHS = 500   #100, 300, 500
TARGET_PER_CLASS = 6000
DEVICE_CONFIG = "GPU"   # or "CPU"

# Torch device
DEVICE = "cuda" if DEVICE_CONFIG == "GPU" and torch.cuda.is_available() else "cpu"

print("Device in use:", DEVICE)


In [ ]:
# CWGAN-GP architecture
class CWGAN_Generator(nn.Module):
    def __init__(self, noise_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim + cond_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
            nn.Sigmoid()
        )

    def forward(self, z, y):
        return self.net(torch.cat([z, y], dim=1))


In [ ]:
class CWGAN_Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim + cond_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x, y):
        return self.net(torch.cat([x, y], dim=1))


In [ ]:
def gradient_penalty(D, real_x, fake_x, y):
    alpha = torch.rand(real_x.size(0), 1).to(DEVICE)
    alpha = alpha.expand_as(real_x)

    interpolates = alpha * real_x + (1 - alpha) * fake_x
    interpolates.requires_grad_(True)

    d_interpolates = D(interpolates, y)
    grads = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(d_interpolates),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


In [ ]:
# Reset the seed so that weight initialisation is reproducible
# regardless of which cells were executed before this one
set_seed(SEED)

# Training
X_tensor = torch.tensor(X_np).to(DEVICE)
y_tensor = torch.tensor(y_np).to(DEVICE)

G = CWGAN_Generator().to(DEVICE)
D = CWGAN_Discriminator().to(DEVICE)

opt_G = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.5, 0.9))
opt_D = torch.optim.Adam(D.parameters(), lr=1e-4, betas=(0.5, 0.9))

LAMBDA_GP = 10
CRITIC_ITER = 5


In [ ]:
for epoch in range(EPOCHS):
    for _ in range(CRITIC_ITER):
        idx = np.random.randint(0, len(X_tensor), 256)

        real_x = X_tensor[idx]
        real_y = y_tensor[idx]

        z = torch.randn(len(real_x), 64).to(DEVICE)
        fake_x = G(z, real_y)

        d_real = D(real_x, real_y).mean()
        d_fake = D(fake_x.detach(), real_y).mean()

        gp = gradient_penalty(D, real_x, fake_x, real_y)
        loss_D = -(d_real - d_fake) + LAMBDA_GP * gp

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

    # ===== Generator =====
    z = torch.randn(len(real_x), 64).to(DEVICE)
    fake_x = G(z, real_y)
    loss_G = -D(fake_x, real_y).mean()

    opt_G.zero_grad()
    loss_G.backward()
    opt_G.step()

    if epoch % 50 == 0:
        print(f"[CWGAN] Epoch {epoch} | D {loss_D:.4f} | G {loss_G:.4f}")


In [ ]:
def generate_cwgan(target_class, n_samples):
    y = np.full((n_samples, 1), target_class)
    y_ohe = ohe.transform(y)

    z = torch.randn(n_samples, 64).to(DEVICE)
    y_tensor = torch.tensor(y_ohe, dtype=torch.float32).to(DEVICE)

    synth = G(z, y_tensor).detach().cpu().numpy()
    return synth


In [ ]:
# Reset the seed so that the generated samples are reproducible
set_seed(SEED)

synthetic_data_list = []

for target_class in sorted(df_cwgan[TARGET_COL].unique()):
    current_count = len(df_cwgan[df_cwgan[TARGET_COL] == target_class])
    num_to_generate = TARGET_PER_CLASS - current_count

    if num_to_generate <= 0:
        continue

    print(f"Generating {num_to_generate} samples for class {target_class}")

    X_synth = generate_cwgan(target_class, num_to_generate)
    df_synth = pd.DataFrame(X_synth, columns=X.columns)
    df_synth[TARGET_COL] = target_class

    synthetic_data_list.append(df_synth)


## Option A - restore valid values

In [ ]:
# ================================================================
# OPTION A - RESTORE VALID VALUES IN THE SYNTHETIC DATA
# ----------------------------------------------------------------
# Runs between generation and saving. The original saving cell
# below is left untouched: synthetic_data_list is replaced by the
# corrected frame, so pd.concat() in that cell simply passes it
# through and writes the corrected data.
# ================================================================
df_synthetic = pd.concat(synthetic_data_list, ignore_index=True)

rep_before = validity_report(df_synthetic, SCHEMA, TARGET_COL, "BEFORE correction")

df_synthetic = apply_schema(df_synthetic, SCHEMA, TARGET_COL)

rep_after = validity_report(df_synthetic, SCHEMA, TARGET_COL, "AFTER correction")

# Keep the before/after comparison for the paper
comparison = rep_before.merge(
    rep_after, on=["Column", "Type", "Levels"], suffixes=(" before", " after")
)
display(comparison)

# The saving cell that follows now receives the corrected data
synthetic_data_list = [df_synthetic]

print("\nCorrected synthetic rows: {:,}".format(len(df_synthetic)))


In [ ]:
# ================================================================
# BUILD THE COMBINED TRAINING SET
# ----------------------------------------------------------------
# Original training records plus the synthetic rows that the
# Option A cell corrected. Written out by the export cell below as
#   <stem>.csv                    (scenario 2)
# while df_synthetic alone becomes
#   <stem>_synthetic_only.csv     (scenario 3)
# ================================================================
df_augmented = pd.concat([df_cwgan, df_synthetic], ignore_index=True)

print("Original training rows :", f"{len(df_cwgan):,}")
print("Synthetic rows         :", f"{len(df_synthetic):,}")
print("Combined rows          :", f"{len(df_augmented):,}")

print("\nClass distribution of the combined training set:")
print(df_augmented[TARGET_COL].value_counts().sort_index())


In [ ]:
# ================================================================
# STANDARDISED EXPORT
# ----------------------------------------------------------------
# Every augmentation notebook writes the same pair of files, so the
# classification notebooks can address them with one naming rule:
#
#   <model>_<device>_epoch<E>_augment<T>_seed<S>.csv
#         df_train + synthetic        -> scenario 2
#   <model>_<device>_epoch<E>_augment<T>_seed<S>_synthetic_only.csv
#         synthetic rows only         -> scenario 3, and notebook 06
#
# df_train.csv is scenario 1 and is not rewritten here.
# ================================================================
MODEL_TAG = "cwgan"
DEVICE_TAG = "gpu" if DEVICE == "cuda" else "cpu"
STEM = f"cwgan_{DEVICE_TAG}_epoch{EPOCHS}_augment{TARGET_PER_CLASS}_seed{SEED}"

COMBINED_FILE = f"{STEM}.csv"
SYNTHETIC_ONLY_FILE = f"{STEM}_synthetic_only.csv"

# ---- checks before anything is written -------------------------
assert len(df_cwgan) + len(df_synthetic) == len(df_augmented), (
    "Combined frame is not original + synthetic: "
    f"{len(df_cwgan)} + {len(df_synthetic)} != {len(df_augmented)}")
assert list(df_augmented.columns) == list(df_synthetic.columns), \
    "Column order differs between the combined and the synthetic frame."
assert df_augmented.isna().sum().sum() == 0, "NaN found in the combined frame."
assert df_synthetic.isna().sum().sum() == 0, "NaN found in the synthetic frame."

counts = df_augmented[TARGET_COL].value_counts().sort_index()

print("Original training rows :", f"{len(df_cwgan):,}")
print("Synthetic rows         :", f"{len(df_synthetic):,}")
print("Combined rows          :", f"{len(df_augmented):,}")
print("Per-class counts       :", counts.tolist())

if counts.nunique() == 1:
    print("All classes balanced at", counts.iloc[0])
else:
    print("WARNING - the classes are not balanced.")

df_augmented.to_csv(COMBINED_FILE, index=False)
df_synthetic.to_csv(SYNTHETIC_ONLY_FILE, index=False)

print("\nSaved:")
print("  ", COMBINED_FILE)
print("       scenario 2 - df_train + synthetic")
print("  ", SYNTHETIC_ONLY_FILE)
print("       scenario 3 - synthetic only, and the input for notebook 06")

files.download(COMBINED_FILE)
files.download(SYNTHETIC_ONLY_FILE)
